# Content-Based Movie Recommendation System

**Original project:** 2022  
**Portfolio version:** curated for GitHub

This notebook implements a content-based movie recommender using movie metadata.
Text features including genres, keywords, tagline, cast, and director are combined,
transformed with TF-IDF, and compared using cosine similarity.

The underlying analytical approach is preserved from the original 2022 project,
while the presentation has been cleaned for portfolio use.

## 1. Imports

In [ ]:
import difflib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load Movie Metadata

In [ ]:
movies_data = pd.read_csv("../data/movies_metadata.csv")
movies_data.head()

The recommender uses five descriptive movie features:

- genres
- keywords
- tagline
- cast
- director

In [ ]:
selected_features = ["genres", "keywords", "tagline", "cast", "director"]

for feature in selected_features:
    movies_data[feature] = movies_data[feature].fillna("")

combined_features = movies_data[selected_features].agg(" ".join, axis=1)

## 3. TF-IDF Feature Representation

In [ ]:
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(combined_features)

print("Movies:", feature_vectors.shape[0])
print("TF-IDF features:", feature_vectors.shape[1])

## 4. Cosine Similarity

In [ ]:
similarity = cosine_similarity(feature_vectors)
print("Similarity matrix shape:", similarity.shape)

## 5. Recommendation Function

In [ ]:
def recommend_movies(movie_name, n_recommendations=10):
    titles = movies_data["title"].tolist()
    matches = difflib.get_close_matches(movie_name, titles, n=1)

    if not matches:
        return pd.DataFrame(columns=["title", "similarity_score"])

    close_match = matches[0]
    movie_index = movies_data.loc[
        movies_data["title"] == close_match, "index"
    ].iloc[0]

    scores = list(enumerate(similarity[movie_index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommendations = []
    for index, score in scores:
        title = movies_data.loc[movies_data["index"] == index, "title"]
        if title.empty:
            continue
        title = title.iloc[0]
        if title == close_match:
            continue
        recommendations.append((title, score))
        if len(recommendations) == n_recommendations:
            break

    return pd.DataFrame(
        recommendations,
        columns=["title", "similarity_score"]
    )

## 6. Example: *Iron Man*

In [ ]:
recommend_movies("Iron Man", n_recommendations=10)

### Interpretation

The content-based model recommends movies with similar descriptive metadata.
For *Iron Man*, the original 2022 implementation produced closely related superhero
titles such as *Iron Man 2*, *Iron Man 3*, *The Avengers*, and
*Captain America: Civil War*.